[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begelb/latent_dynamics/blob/paper/notebooks/02_leslie3d_example1.ipynb)

# Three-dimensional Leslie model

This example compares four views of the same dynamics: a direct
three-dimensional reference, a fine two-dimensional latent Morse graph,
a connection-complete coarsening, and a lower fixed grid resolution.
The fine graph has an extra minimal node; merging nodes 4 and 5 restores
the two-attractor structure without discarding the connecting cells.

Everything below is loaded from checksummed release bundles. The paper
training design is 3,200 trajectories with 800 held out for validation.


In [ ]:
# Colab keeps a checkout so the artifact manifest remains available.
import os
import shutil
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/latent_dynamics")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "paper",
             "https://github.com/begelb/latent_dynamics.git", str(root)],
            check=True,
        )
    if shutil.which("dot") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "graphviz"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "cmgdb==1.3.3+fork.3",
         "--find-links", "https://github.com/bernardorivas/CMGDB/releases/expanded_assets/v1.3.3%2Bfork.3"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
    os.chdir(root)


## Load the direct and latent computations


In [ ]:
import json
from latentdynamics.replay import fetch_bundle, load_experiment, show_image

direct = fetch_bundle("original_leslie3d_reference")
latent = load_experiment("leslie3d_example1_replay")
study = latent.seed_dir.parent / "study" / "fixed22_vs_adaptive"


## Direct reference


In [ ]:
base = direct / "absorbing_B_uniform_level33_recurrent_closure"
show_image(base / "paper_figure_pruned" / "morse_graph.png", width=560)
show_image(base / "cubical_3d_level24_display_cover" / "morse_sets_cubical_3d.png", width=660)


## Fine latent result, connection-complete coarsening, and uniform level 22


In [ ]:
show_image(latent.morse_dir / "morse_graph.png", width=560)
panels = study / "paper_ready_no_legend"
for name in ["adaptive_merged_4_5_morse_sets_no_legend.png",
             "uniform_22_nontrivial_morse_sets_no_legend.png"]:
    show_image(panels / name, width=660)


In [ ]:
merged = json.loads((study / "adaptive_23_23_27" / "merged_4_5" / "result.json").read_text())
uniform = json.loads((study / "uniform_22_22_22" / "result.json").read_text())
print("fine minimal:", merged["fine_graph"]["minimal"])
print("coarsened minimal:", merged["coarse_graph"]["minimal"],
      "| merged cells:", merged["connection_completion"]["merged_cells"])
print("uniform-22:", uniform["fixed_graph"]["node_count"], "nodes; minimal",
      uniform["fixed_graph"]["minimal"])


Expected replay invariants are three fine minimal nodes, two after the
322-cell connection-complete merge, and 24 nodes with two minimal nodes
at uniform level 22. Fresh retraining is stochastic and does not
reliably reproduce the fine graph's third attractor.
